# BioVU CHIP EWAS: Results and Activity Score Validation
**Pershad & Zhao et al., Nature Medicine, 2025**

This notebook covers:
1. Aggregating scattered EWAS chunk outputs from Terra
2. Volcano plot visualization of EWAS results (beta vs. –log10 P)
3. Application of TET2 and DNMT3A Activity Scores to BioVU samples
4. Score comparison across mutation subtypes (pairwise Wilcoxon tests)

EWAS were run using `wdl/run-logistic-ewas.wdl` and `wdl/run-logistic-dmr-ewas.wdl` on Terra.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import stats
from scipy.stats import norm

## 1. Aggregate scattered EWAS outputs

Each EWAS is run in parallel across CpG chunks on Terra. Here we combine the chunk outputs
into a single summary statistics file per analysis.

In [ ]:
%%bash
# Helper function: collect scattered chunk outputs and concatenate
# Usage: collect_ewas <gcs_submission_path> <output_dir> <output_prefix>

collect_ewas() {
    local gcs_path=$1
    local out_dir=$2
    local prefix=$3

    mkdir -p "$out_dir"
    gsutil -m cp "${gcs_path}/call-ewas_logistic_chunk/shard-*/**chunk_*_results.tsv" "${out_dir}/"

    # Concatenate: header from first chunk, data from all chunks
    head -n 1 "${out_dir}/$(ls ${out_dir} | head -1)" > "${out_dir}/${prefix}_combined_results.tsv"
    tail -n +2 -q "${out_dir}/chunk_"*.tsv >> "${out_dir}/${prefix}_combined_results.tsv"

    echo "${prefix}: $(wc -l < ${out_dir}/${prefix}_combined_results.tsv) CpG sites"
}

# Run for each analysis (replace GCS paths with your Terra submission paths)
collect_ewas "gs://fc-secure-WORKSPACE/submissions/SUBMISSION_ID_TET2PLOF/run_scattered_logistic_ewas/RUN_ID" \
             "tet2_plof_ewas_summ_stats" "tet2_plof"
collect_ewas "gs://fc-secure-WORKSPACE/submissions/SUBMISSION_ID_D3A/run_scattered_logistic_ewas/RUN_ID" \
             "d3a_ewas_summ_stats" "d3a"

In [ ]:
# Load EWAS summary statistics
tet2_plof_df = pd.read_table('tet2_plof_ewas_summ_stats/tet2_plof_combined_results.tsv')
tet2_miss_df  = pd.read_table('tet2_miss_ewas_summ_stats/tet2_miss_combined_results.tsv')
d3a_r882_df   = pd.read_table('d3a_r882_ewas_summ_stats/d3a_r882_combined_results.tsv')
d3a_nonr882_df = pd.read_table('d3a_nonr882_ewas_summ_stats/d3a_nonr882_combined_results.tsv')

# Load sliding window DMR results
tet2_dmr_df   = pd.read_table('tet2_ewas_sliding_window_dmrs.tsv')
d3a_dmr_df    = pd.read_table('d3a_ewas_sliding_window_dmrs.tsv')

print(f"TET2 pLoF EWAS: {len(tet2_plof_df):,} CpG sites")
print(f"DNMT3A R882 EWAS: {len(d3a_r882_df):,} CpG sites")

## 2. Volcano plot visualization

Volcano plots show beta coefficients (x-axis) vs. –log10(P) (y-axis) for each CpG site.
Significant sites (Bonferroni-corrected P < 0.05, |beta| > 0.1) are colored by direction.
Style follows Nature Genetics formatting guidelines.

In [ ]:
def plot_ewas_volcano(df,
                      beta_col='beta',
                      pval_col='p_value',
                      beta_threshold=0.1,
                      pval_threshold=None,
                      title='CHIP EWAS Volcano Plot',
                      figsize=(8, 6),
                      save_path=None,
                      dpi=300):
    """
    Volcano plot of EWAS results: beta coefficient vs. –log10(P).
    Significant CpGs are colored red (hypermethylated) or blue (hypomethylated).

    Parameters
    ----------
    df             : DataFrame with EWAS summary statistics
    beta_col       : column name for beta coefficients
    pval_col       : column name for p-values
    beta_threshold : minimum |beta| for significance annotation (default 0.1)
    pval_threshold : p-value threshold; if None, uses Bonferroni (0.05 / n_tests)
    title          : plot title
    figsize        : figure size in inches
    save_path      : path to save figure (None = display only)
    dpi            : resolution for saved figure
    """
    rcParams.update({
        'font.family': 'DejaVu Sans', 'font.size': 12,
        'axes.linewidth': 1.0, 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.edgecolor': 'black', 'xtick.direction': 'out', 'ytick.direction': 'out',
        'xtick.major.size': 4, 'ytick.major.size': 4,
        'figure.facecolor': 'white', 'axes.facecolor': 'white'
    })

    if pval_threshold is None:
        pval_threshold = 0.05 / len(df)  # Bonferroni correction

    log_pvals = -np.log10(df[pval_col].replace(0, 1e-300))
    betas     = df[beta_col]
    pval_thresh_log = -np.log10(pval_threshold)

    # Color: red = significant hypermethylation, blue = significant hypomethylation, gray = NS
    sig_mask  = (log_pvals >= pval_thresh_log) & (betas.abs() >= beta_threshold)
    colors    = np.where(sig_mask & (betas > 0), '#d62728',
                np.where(sig_mask & (betas < 0), '#1f77b4', '#b0b0b0'))

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    ax.scatter(betas, log_pvals, c=colors, s=10, alpha=0.6, edgecolors='none', rasterized=True)

    # Significance threshold lines
    ax.axhline(pval_thresh_log, color='#555555', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axvline(-beta_threshold, color='#555555', linestyle='--', linewidth=0.8, alpha=0.7)
    ax.axvline( beta_threshold, color='#555555', linestyle='--', linewidth=0.8, alpha=0.7)

    # Annotate counts
    n_hyper = int(((betas > beta_threshold)  & sig_mask).sum())
    n_hypo  = int(((betas < -beta_threshold) & sig_mask).sum())
    ax.text(0.98, 0.97, f'Hypermeth.: {n_hyper:,}', transform=ax.transAxes,
            ha='right', va='top', color='#d62728', fontsize=10)
    ax.text(0.02, 0.97, f'Hypometh.: {n_hypo:,}', transform=ax.transAxes,
            ha='left', va='top', color='#1f77b4', fontsize=10)

    ax.set_xlabel('Beta coefficient (methylation difference)', fontsize=12)
    ax.set_ylabel('–log\u2081\u2080(P)', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    plt.show()
    return fig, ax

In [ ]:
# TET2 pLoF EWAS volcano plot
fig, ax = plot_ewas_volcano(
    tet2_plof_df,
    beta_threshold=0.05,
    title='TET2 pLoF vs. Control — CpG-level EWAS',
    save_path='figures/tet2_plof_ewas_volcano.png'
)

In [ ]:
# DNMT3A R882 EWAS volcano plot
fig, ax = plot_ewas_volcano(
    d3a_r882_df,
    beta_threshold=0.05,
    title='DNMT3A R882 vs. Control — CpG-level EWAS',
    save_path='figures/d3a_r882_ewas_volcano.png'
)

## 3. Apply Activity Scores in BioVU and compare across mutation subtypes

Activity scores were trained in the Framingham Heart Study and applied here to BioVU.
Scores are standardized to the non-CHIP control distribution (mean=0, SD=1).
See `activity_score_construction.R` for the full training and scoring pipeline.

In [ ]:
from scipy.stats import wilcoxon, ranksums
from itertools import combinations

# Load activity scores computed from R pipeline
scores_df = pd.read_table('biovu_activity_scores.tsv')
# Expected columns: sample_id, mutation_class, tet2_activity_score, dnmt3a_activity_score, vaf

print("Sample counts by mutation class:")
print(scores_df['mutation_class'].value_counts())

In [ ]:
def compare_scores_by_group(df, score_col, group_col='mutation_class',
                             ordered_groups=None, alpha=0.05):
    """
    Pairwise Wilcoxon rank-sum tests comparing activity scores across mutation classes.
    Returns a summary DataFrame with median scores and Bonferroni-corrected p-values.
    """
    groups = ordered_groups or df[group_col].unique().tolist()
    n_comparisons = len(list(combinations(groups, 2)))

    summary = df.groupby(group_col)[score_col].agg(['median', 'count']).loc[groups]
    summary.columns = ['median_score', 'n']

    print(f"\n{score_col} — Median scores by group:")
    print(summary.to_string())
    print(f"\nPairwise Wilcoxon tests (Bonferroni correction for {n_comparisons} comparisons):")

    results = []
    for g1, g2 in combinations(groups, 2):
        x = df.loc[df[group_col] == g1, score_col].dropna()
        y = df.loc[df[group_col] == g2, score_col].dropna()
        stat, p = ranksums(x, y)
        p_adj = min(p * n_comparisons, 1.0)
        results.append({'group1': g1, 'group2': g2, 'statistic': stat, 'p_value': p, 'p_bonferroni': p_adj})
        sig = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01 else ('*' if p_adj < alpha else 'ns'))
        print(f"  {g1} vs {g2}: P = {p:.2e} (adj. P = {p_adj:.2e}) {sig}")

    return pd.DataFrame(results)

# TET2 Activity Score comparison across mutation subtypes
tet2_groups = ['Early pLoF', 'Late pLoF', 'Missense', 'Control']
tet2_pairwise = compare_scores_by_group(
    scores_df, 'tet2_activity_score',
    ordered_groups=tet2_groups
)

In [ ]:
# DNMT3A Activity Score comparison across mutation subtypes
d3a_groups = ['R882', 'pLoF', 'Non-R882 missense', 'Control']
d3a_pairwise = compare_scores_by_group(
    scores_df, 'dnmt3a_activity_score',
    ordered_groups=d3a_groups
)

In [ ]:
def plot_activity_scores_by_group(df, score_col, group_col='mutation_class',
                                   ordered_groups=None, palette=None,
                                   title='Activity Score by Mutation Class',
                                   ylabel='Activity Score (z-score)',
                                   save_path=None, dpi=300):
    """
    Strip plot with median bars comparing activity scores across mutation subtypes.
    """
    groups = ordered_groups or df[group_col].unique().tolist()
    if palette is None:
        palette = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8']

    rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

    fig, ax = plt.subplots(figsize=(len(groups) * 1.8, 5), dpi=dpi)

    for i, (group, color) in enumerate(zip(groups, palette)):
        vals = df.loc[df[group_col] == group, score_col].dropna()
        # Jittered strip
        jitter = np.random.normal(0, 0.08, size=len(vals))
        ax.scatter(np.full(len(vals), i) + jitter, vals,
                   c=color, s=15, alpha=0.5, edgecolors='none', zorder=2)
        # Median bar
        ax.plot([i - 0.3, i + 0.3], [vals.median(), vals.median()],
                color='black', linewidth=2.5, zorder=3)

    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups, rotation=20, ha='right')
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    plt.show()
    return fig, ax


# TET2 Activity Score by mutation class
plot_activity_scores_by_group(
    scores_df, 'tet2_activity_score',
    ordered_groups=['Early pLoF', 'Late pLoF', 'Missense', 'Control'],
    palette=['#d62728', '#ff7f0e', '#2ca02c', '#7f7f7f'],
    title='TET2 Activity Score by Mutation Subtype (BioVU + CHIVE)',
    save_path='figures/tet2_activity_score_by_mutation_type.png'
)

In [ ]:
# Check correlation between activity score and VAF (expected: modest, <5% variance explained)
from scipy.stats import spearmanr

for gene, score_col in [('TET2', 'tet2_activity_score'), ('DNMT3A', 'dnmt3a_activity_score')]:
    chip_df = scores_df[scores_df['mutation_class'] != 'Control'].dropna(subset=[score_col, 'vaf'])
    rho, p = spearmanr(chip_df[score_col], chip_df['vaf'])
    r2 = rho ** 2
    print(f"{gene} Activity Score vs. VAF: Spearman rho = {rho:.3f}, P = {p:.2e}, R² = {r2:.3f}")
    print(f"  → VAF explains {r2 * 100:.1f}% of variance in {gene} activity score")